# Lab 3: Sequential Incident Triage and Crew Planning

**Required · 45 minutes · Level 200**

Use an explicit two-stage workflow: a triage agent structures the incident,
then a planner proposes a diagnostic dispatch plan.

## Learning objectives

- Choose a workflow when order must be controlled.
- Build a sequential orchestration with two specialized agents.
- Inspect the final workflow artifact.

This is advisory training content. It does not operate grid equipment or
dispatch a real crew.

In [ ]:
import os
import re

from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)


def safe_name(value: str, *, max_length: int = 40) -> str:
    value = re.sub(r"[^a-z0-9-]+", "-", value.lower()).strip("-")
    value = re.sub(r"-+", "-", value)
    if not value:
        raise ValueError("Resource namespace must contain a letter or number.")
    return value[:max_length].rstrip("-")


raw_namespace = (
    os.getenv("WORKSHOP_RESOURCE_NAMESPACE")
    or os.getenv("WORKSHOP_TEAM_ID")
    or os.getenv("WORKSHOP_PARTICIPANT_ID")
)
if not raw_namespace:
    raise ValueError(
        "Set WORKSHOP_RESOURCE_NAMESPACE (preferred), WORKSHOP_TEAM_ID, "
        "or WORKSHOP_PARTICIPANT_ID before running workshop labs."
    )

RESOURCE_NAMESPACE = safe_name(raw_namespace)
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL = os.environ["FOUNDRY_MODEL"]

print(f"Namespace: {RESOURCE_NAMESPACE}")
print(f"Model: {MODEL}")

In [ ]:
from agent_framework.foundry import FoundryChatClient
from agent_framework.orchestrations import SequentialBuilder
from azure.identity import AzureCliCredential

credential = AzureCliCredential()
client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL,
    credential=credential,
)

triage_agent = client.as_agent(
    name=f"incident-triage-{RESOURCE_NAMESPACE}",
    instructions=(
        "Extract only stated facts from the synthetic incident. Produce "
        "FACTS, UNKNOWNS, and SEVERITY RATIONALE. Do not suggest switching."
    ),
)
crew_planner = client.as_agent(
    name=f"crew-planner-{RESOURCE_NAMESPACE}",
    instructions=(
        "Use the preceding triage. Produce PLAN, TEAM, REQUIRED EVIDENCE, "
        "and HUMAN APPROVAL. Recommend diagnostic work only. Explicitly say "
        "that a qualified operator must approve any operational action."
    ),
)

workflow = SequentialBuilder(
    participants=[triage_agent, crew_planner],
    output_from=[crew_planner],
).build()

## Participant task

Add one relevant unknown to the synthetic incident description. Predict how it
should affect the proposed plan.

In [ ]:
# TODO(participant): add a synthetic unknown such as relay evidence availability.
INCIDENT = '''
Training incident INC-1042:
- transformer TR-104 produced two temperature alarms in 20 minutes
- telemetry is still available
- no customer outage is confirmed
- West Operations owns this asset class
'''

events = await workflow.run(INCIDENT)
outputs = events.get_outputs()
assert outputs, "The workflow produced no final output."

final_output = outputs[0]
final_text = getattr(final_output, "text", None) or str(final_output)
print(final_text)

## Deterministic success check

In [ ]:
assert triage_agent.name != crew_planner.name
assert final_text.strip(), "The planner output is empty."
assert RESOURCE_NAMESPACE in triage_agent.name
assert RESOURCE_NAMESPACE in crew_planner.name
print("PASS — both names are isolated and the final workflow artifact exists.")

## Optional extension

Enable a human-in-the-loop pause before the planner when the installed
orchestration version supports `with_request_info`.

**Expected artifact:** a fact-first triage followed by an approval-aware crew plan.